# 📋 Ejercicio Clase 1 — Validación Temporal y Backtesting: No-Renovación de Auto
## Diplomado ML en Seguros · Subtema 4

---

## Contexto

**SeguraMax** entrenó un modelo GBM para predecir no-renovaciones **en 2021** y lo puso en producción. Desde entonces, el modelo lleva **3 años sin re-entrenarse**. El mercado cambió: los comparadores digitales hicieron trivial cambiar de aseguradora, y los incrementos de prima — que antes retenían clientes — ahora los expulsan.

El director de Retención pregunta: *"¿Nuestro modelo sigue siendo válido? ¿Cuánto valor habría generado? ¿Vale la pena re-entrenarlo?"*

Tu trabajo responde esas tres preguntas.

---

## Lo que practicarás

| Parte | Concepto |
|-------|----------|
| 2 | Demostrar que K-Fold da un AUC falso |
| 3 | Evaluar el modelo congelado — ver el drift real |
| 4 | Walk-Forward — cuantificar el beneficio de re-entrenar |
| 5 | Visualizar el drift estructural |
| 6 | Backtesting económico: ¿cuánto dinero habría generado? |
| 7 | Preguntas de reflexión y decisión final |

---

## Costos del negocio

| Parámetro | Valor |
|-----------|-------|
| C_FP — llamada de retención innecesaria | $400 MXN |
| C_FN — cliente perdido sin contactar | $11,200 MXN (prima anual media) |
| Probabilidad de retener si se llama | 32% |
| τ por costos | **calcúlalo en la primera celda** |

**No cambies `random_state=2024`.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix

np.random.seed(2024)

C_FP  = 400
C_FN  = 11_200
P_RET = 0.32

# 🔧 COMPLETA: calcula τ por costos
tau_c = ...   # fórmula: C_FP / (C_FP + C_FN)

print(f"C_FP = ${C_FP:,}  C_FN = ${C_FN:,}  P_RET = {P_RET}")
print(f"τ por costos = {tau_c:.4f}")
print(f"Ratio C_FN/C_FP = {C_FN/C_FP:.0f}x")
print(f"Con τ = {tau_c:.4f}: alertamos ante cualquier probabilidad > {tau_c*100:.1f}%")

---
## Parte 1 — Portafolio SeguraMax 2020–2024 (NO modificar)

El portafolio tiene **drift estructural simulado**:
- El coeficiente de `incremento_prima_pct` cambia de signo entre 2020 y 2024
- El efecto protector de `uso_app` se debilita cada año

Ejecuta esta celda sin modificar nada.

In [ ]:
# ── Portafolio SeguraMax 2020–2024 — NO MODIFICAR ────────────────────────────
np.random.seed(2024)
ANIOS  = [2020, 2021, 2022, 2023, 2024]
N_ANIO = {2020:460, 2021:495, 2022:510, 2023:480, 2024:505}

registros = []
for anio in ANIOS:
    n   = N_ANIO[anio]
    ac  = np.random.choice(range(21), n,
          p=[.10,.12,.11,.09,.08,.07,.06,.05,.05,
             .04,.04,.03,.03,.03,.02,.02,.02,.01,.01,.01,.01])
    np_ = np.random.choice([1,2,3,4], n, p=[.58,.28,.10,.04])
    inc = np.clip(np.random.exponential(9, n), 0, 42).round(1)
    app = np.random.binomial(1, 0.40, n)
    qx  = np.random.choice([0,1,2,3], n, p=[.77,.17,.05,.01])
    cot = np.random.binomial(1, 0.22 + 0.03*(anio-2020), n)
    can = np.random.choice([0,1,2], n, p=[.50,.30,.20])

    # Drift estructural: coeficientes que cambian con el tiempo
    coef_inc = -0.05 + 0.028*(anio - 2020)  # de -0.05 (2020) a +0.062 (2024)
    coef_app = -0.50 + 0.09 *(anio - 2020)  # app protege menos cada año

    lo = (-1.30 - 0.19*ac - 0.52*np_ + coef_inc*inc + coef_app*app
          + 0.78*qx + 1.05*cot + 0.28*(can==1)
          + np.random.normal(0, 0.62, n))
    ch = (np.random.rand(n) < 1/(1+np.exp(-lo))).astype(int)

    for i in range(n):
        registros.append({'anio':anio, 'anios_cliente':ac[i], 'num_polizas':np_[i],
                          'incremento_prima_pct':inc[i], 'uso_app':app[i],
                          'quejas_12m':qx[i], 'competidor_cotizo':cot[i],
                          'canal_venta':can[i], 'no_renueva':ch[i]})

df = pd.DataFrame(registros).sort_values('anio').reset_index(drop=True)
FEATURES = ['anios_cliente','num_polizas','incremento_prima_pct',
            'uso_app','quejas_12m','competidor_cotizo','canal_venta']

print(f"Portafolio SeguraMax: {len(df):,} pólizas 2020–2024")
print()
print(f"  {'Año':>5}  {'n':>5}  {'No renueva':>11}  {'Coef. prima (contexto)':>24}")
coefs = {a: round(-0.05+0.028*(a-2020), 3) for a in ANIOS}
for a in ANIOS:
    s = df[df.anio==a]
    sentido = "prima alta → se QUEDA" if coefs[a] < 0 else "prima alta → se VA"
    print(f"  {a:>5}  {len(s):>5,}  {s.no_renueva.mean()*100:>10.1f}%  {coefs[a]:>+8.3f}  ({sentido})")
print()
print("Nota: el coeficiente de prima cambia de signo entre 2020 y 2024.")
print("El modelo entrenado en 2020–2021 aprende la dirección INCORRECTA para 2024.")

---
## Parte 2 — K-Fold estándar: el AUC que la dirección vio

El equipo de datos validó el modelo con K-Fold y reportó AUC = 0.72.
Demuestra que ese número es ambiguo — no corresponde a ningún año real.

### 🔧 2.1 — Calcula el AUC con K-Fold estándar

In [ ]:
# 🔧 COMPLETA: K-Fold estándar con 5 folds, shuffle=True, random_state=2024
modelo = Pipeline([
    ('sc', StandardScaler()),
    ('m',  GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                       learning_rate=0.08, random_state=2024))
])
X_all = df[FEATURES].values
y_all = df['no_renueva'].values

# --- TU CÓDIGO AQUÍ ---
kf      = ...   # StratifiedKFold(...)
scores  = ...   # cross_val_score(...)
auc_kf  = ...   # promedio

print(f"AUC K-Fold (5 folds): {auc_kf:.4f}")
print(f"Std entre folds:      {scores.std():.4f}")
print()
print("¿A qué año corresponde este AUC?")
print("Responde en la celda de texto siguiente.")

### 📝 Pregunta 2.1

K-Fold mezcla pólizas de 2020 con 2024 en el mismo fold. En este portafolio,
el coeficiente de `incremento_prima_pct` tiene **signo opuesto** en 2020 vs 2024.

**¿Qué aprende el modelo cuando los dos años están mezclados en el mismo fold?**
¿Por qué ese AUC no representa lo que pasará en producción?

> _Escribe tu respuesta aquí_

---
## Parte 3 — El modelo que está en producción: congelado desde 2021

El modelo de SeguraMax fue entrenado con datos de **2020 y 2021** y lleva
**3 años sin re-entrenarse**. Evalúa ese modelo congelado año a año para
ver cómo se degrada.

### 🔧 3.1 — Entrena el modelo congelado y evalúalo en 2022, 2023 y 2024

In [ ]:
# 🔧 COMPLETA: entrena el modelo solo con datos de 2020 y 2021
# Luego evalúa ese mismo modelo (sin re-entrenar) en 2022, 2023 y 2024

mask_train_ini = df['anio'].isin([2020, 2021])
X_train_ini    = df.loc[mask_train_ini, FEATURES].values
y_train_ini    = df.loc[mask_train_ini, 'no_renueva'].values

pipe_congelado = Pipeline([
    ('sc', StandardScaler()),
    ('m',  GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                       learning_rate=0.08, random_state=2024))
])
# 🔧 ENTRENA el modelo congelado
# --- TU CÓDIGO AQUÍ ---

print("MODELO CONGELADO — entrenado en 2020–2021, nunca re-entrenado:")
print()
print(f"  {'Año test':>9}  {'AUC':>8}  {'Tasa real':>11}  {'Diagnóstico'}")
print(f"  {'-'*55}")

aucs_congelado = []
for anio_test in [2022, 2023, 2024]:
    mask_test = df['anio'] == anio_test
    X_test    = df.loc[mask_test, FEATURES].values
    y_test    = df.loc[mask_test, 'no_renueva'].values

    # 🔧 PREDICE con el modelo congelado
    # --- TU CÓDIGO AQUÍ ---
    y_prob = ...
    auc    = roc_auc_score(y_test, y_prob)
    aucs_congelado.append(auc)
    tasa   = y_test.mean() * 100

    if auc >= 0.70:   diag = "✅ Aceptable"
    elif auc >= 0.65: diag = "⚠️  Degradación — investigar"
    else:             diag = "🔴 Drift significativo — re-entrenar"

    print(f"  {anio_test:>9}  {auc:>8.4f}  {tasa:>10.1f}%  {diag}")

print()
print("¿El AUC sube o baja año a año?")
print("¿Qué dice eso sobre el modelo que está en producción?")

### 📝 Pregunta 3.1

El modelo fue entrenado cuando `incremento_prima_pct` tenía coeficiente negativo
(prima alta → cliente se queda). En 2024 ese coeficiente es positivo (prima alta → cliente se va).

**¿Qué error concreto comete el modelo congelado en 2024?**
¿Alerta a los clientes incorrectos? ¿Cuál es el impacto para Retención?

> _Escribe tu respuesta aquí_

---
## Parte 4 — Walk-Forward: ¿cuánto ayudaría re-entrenar cada año?

El Walk-Forward simula: *"si hubiéramos re-entrenado el modelo cada año con
todos los datos históricos disponibles, ¿cómo habría sido el AUC?"*

### 🔧 4.1 — Implementa el loop de Walk-Forward

In [ ]:
# 🔧 COMPLETA: Walk-Forward para 2022, 2023, 2024
# En cada ventana: train = todos los años ANTERIORES al test, test = ese año

ANIOS_TEST    = [2022, 2023, 2024]
resultados_wf = []

print("COMPARACIÓN: Modelo Congelado vs Walk-Forward (re-entrenamiento anual)")
print()
print(f"  {'Año':>5}  {'Train WF':>14}  {'AUC Congelado':>15}  "
      f"{'AUC Walk-Fwd':>14}  {'Ganancia':>10}")
print(f"  {'-'*66}")

for i, anio_test in enumerate(ANIOS_TEST):
    # 🔧 Define train y test
    mask_train = ...   # df['anio'] < anio_test
    mask_test  = ...   # df['anio'] == anio_test

    X_train = df.loc[mask_train, FEATURES].values
    y_train = df.loc[mask_train, 'no_renueva'].values
    X_test  = df.loc[mask_test,  FEATURES].values
    y_test  = df.loc[mask_test,  'no_renueva'].values

    # 🔧 Entrena el modelo Walk-Forward y predice
    pipe_wf = Pipeline([
        ('sc', StandardScaler()),
        ('m',  GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                           learning_rate=0.08, random_state=2024))
    ])
    # --- TU CÓDIGO AQUÍ ---
    y_prob = ...
    auc_wf = roc_auc_score(y_test, y_prob)

    ganancia = auc_wf - aucs_congelado[i]
    anios_tr = sorted(df.loc[mask_train,'anio'].unique())

    resultados_wf.append({'anio':anio_test, 'auc_wf':auc_wf,
                           'auc_congelado':aucs_congelado[i],
                           'y_test':y_test, 'y_prob':y_prob})

    print(f"  {anio_test:>5}  {anios_tr[0]}–{anios_tr[-1]}  "
          f"{aucs_congelado[i]:>15.4f}  {auc_wf:>14.4f}  {ganancia:>+9.4f}")

print()
print(f"  K-Fold (número ambiguo): {auc_kf:.4f}")
print()
print("¿El Walk-Forward supera al modelo congelado en 2024?")
print("¿Cuánto habría ganado el AUC con re-entrenamiento anual?")

---
## Parte 5 — Visualización: el drift visto desde tres ángulos

### 🔧 5.1 — Tres paneles

In [ ]:
# 🔧 COMPLETA: figura con 3 paneles
# Panel 1: AUC congelado (rojo punteado) vs Walk-Forward (verde) vs K-Fold (gris punteado)
# Panel 2: Tasa real de no-renovación por año (barras)
# Panel 3: Relación entre incremento_prima_pct y churn por año
#           (para 2020 y 2024: agrupa por quintiles de inc y muestra la tasa de no-renueva)
#           Este panel muestra visualmente el cambio de signo

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Drift del Modelo SeguraMax — 3 Perspectivas', fontweight='bold')

años_p      = [r['anio']          for r in resultados_wf]
aucs_wf_p   = [r['auc_wf']        for r in resultados_wf]
aucs_cong_p = [r['auc_congelado'] for r in resultados_wf]

# Panel 1
ax = axes[0]
# --- TU CÓDIGO AQUÍ ---

# Panel 2
ax = axes[1]
# --- TU CÓDIGO AQUÍ ---

# Panel 3: cambio de signo de la relación prima-churn
ax = axes[2]
# Pista: para 2020 y 2024, agrupa por quintiles de inc y grafica la tasa de churn
# Para ver el drift: en 2020 la línea debería BAJAR, en 2024 debería SUBIR
# --- TU CÓDIGO AQUÍ ---

plt.tight_layout()
plt.savefig('ej1_drift.png', dpi=150, bbox_inches='tight')
plt.show()
print("Panel 3 es el más importante: muestra el drift ESTRUCTURAL.")
print("La misma variable (incremento_prima_pct) predice en dirección OPUESTA según el año.")

---
## Parte 6 — Backtesting: ¿las decisiones del modelo fueron correctas en producción?

Hasta aquí evaluaste el modelo con métricas estáticas (AUC, Walk-Forward).
El **backtesting** hace algo diferente: toma el **modelo congelado que está en producción**,
lo aplica período a período sobre datos reales históricos, y mide si las decisiones
que habría tomado resultaron correctas cuando se reveló la realidad.

### La diferencia con lo que hiciste antes

| Lo que hiciste en Partes 3–4 | Backtesting (esta parte) |
|------------------------------|--------------------------|
| Compara AUC de congelado vs WF | El modelo es fijo: el congelado de producción |
| Usa años completos | Aplica el modelo **trimestre a trimestre** |
| Mide discriminación global | Mide aciertos/fallos **cuando se reveló la realidad** |
| No dice si las alertas fueron rentables | Calcula el **valor neto realizado** periodo a periodo |

### 🔧 6.1 — Backtesting trimestral del modelo congelado (2022–2024)

Simula que cada trimestre el modelo de SeguraMax generó alertas sobre las pólizas
que vencían ese período. Al final del trimestre, la realidad reveló quién sí se fue.
Mide si las alertas fueron correctas y cuánto valor generaron.

In [ ]:
# 🔧 COMPLETA: backtesting trimestral del modelo congelado
#
# Para cada trimestre de 2022–2024:
#   1. Toma la cohorte de pólizas de ese trimestre (ya están asignadas)
#   2. Aplica el modelo CONGELADO (pipe_congelado) — el mismo siempre, sin re-entrenar
#   3. Genera alertas: y_alerta = (y_prob >= tau_c).astype(int)
#   4. Mide: VP, FP, FN, Precisión, Recall
#   5. Calcula el valor neto REALIZADO ese trimestre
#      valor_neto = round(VP * P_RET) * C_FN - (VP + FP) * C_FP
#   6. Guarda el Brier Score (calibración del modelo)
#      from sklearn.metrics import brier_score_loss
#      brier = brier_score_loss(y_real, y_prob)

from sklearn.metrics import brier_score_loss

# Asignación de trimestres (NO modificar — define las cohortes)
np.random.seed(42)
ANIOS_BT = [2022, 2023, 2024]
resultados_bt = []

for anio in ANIOS_BT:
    mask_año = df['anio'] == anio
    df_año   = df[mask_año].copy()
    df_año['trimestre'] = np.random.choice([1,2,3,4], len(df_año), p=[.28,.26,.24,.22])

    for q in [1, 2, 3, 4]:
        cohort = df_año[df_año['trimestre'] == q]
        if len(cohort) < 20:
            continue

        X_c    = cohort[FEATURES].values
        y_real = cohort['no_renueva'].values

        # 🔧 COMPLETA: aplica el modelo CONGELADO y calcula todas las métricas
        # --- TU CÓDIGO AQUÍ ---
        y_prob   = ...  # pipe_congelado.predict_proba(X_c)[:,1]
        y_alerta = ...  # umbral tau_c

        if y_alerta.sum() == 0 or y_real.sum() == 0:
            continue

        # 🔧 CALCULA: AUC, VP, FP, FN, Precisión, Recall, Brier, Valor neto
        auc   = ...
        cm_   = confusion_matrix(y_real, y_alerta)
        vn_, fp_, fn_, vp_ = cm_.ravel()
        prec  = ...  # vp_ / (vp_ + fp_)
        rec   = ...  # vp_ / (vp_ + fn_)
        brier = brier_score_loss(y_real, y_prob)

        retenidos  = round(vp_ * P_RET)
        vnet       = ...  # retenidos*C_FN - (vp_+fp_)*C_FP

        # Score promedio de los que SÍ se fueron vs los que NO
        score_pos = y_prob[y_real == 1].mean()
        score_neg = y_prob[y_real == 0].mean()

        resultados_bt.append({
            'periodo': f"{anio}-Q{q}", 'anio': anio, 'trimestre': q,
            'n': len(cohort), 'alertas': y_alerta.sum(), 'reales': y_real.sum(),
            'auc': auc, 'brier': brier,
            'vp': vp_, 'fp': fp_, 'fn': fn_,
            'precision': prec, 'recall': rec,
            'score_pos': score_pos, 'score_neg': score_neg,
            'retenidos': retenidos, 'vnet': vnet,
            'tasa_real': y_real.mean()
        })

df_bt = pd.DataFrame(resultados_bt)

print(f"Backtesting: {len(df_bt)} períodos evaluados (modelo fijo: congelado 2020–2021)")
print()
print(f"  {'Período':>10}  {'n':>5}  {'Alertas':>8}  {'VP':>4}  {'FP':>4}  "
      f"{'FN':>4}  {'Prec':>7}  {'Recall':>7}  {'AUC':>7}  {'Brier':>7}  {'Valor neto':>12}")
print(f"  {'-'*95}")

for _, r in df_bt.iterrows():
    print(f"  {r['periodo']:>10}  {r['n']:>5,}  {r['alertas']:>8,}  {r['vp']:>4}  "
          f"{r['fp']:>4}  {r['fn']:>4}  {r['precision']:>6.1%}  {r['recall']:>6.1%}  "
          f"{r['auc']:>7.3f}  {r['brier']:>7.4f}  ${r['vnet']:>10,.0f}")

print()
print(f"Valor neto TOTAL acumulado: ${df_bt['vnet'].sum():,.0f}")
print(f"AUC promedio en producción: {df_bt['auc'].mean():.4f}")

### 🔧 6.2 — Visualización del backtesting período a período

Esta gráfica es el corazón del backtesting: muestra cómo se comportó el modelo
**real en producción**, trimestre a trimestre, desde 2022 hasta 2024.

In [ ]:
# 🔧 COMPLETA: figura con 3 paneles
# Panel 1: AUC por trimestre — ¿se mantuvo la discriminación?
#   - Grafica df_bt['auc'] por período, con colores distintos por año
#   - Agrega línea horizontal con el AUC promedio
# Panel 2: Separación de scores — ¿el modelo sigue separando positivos de negativos?
#   - Grafica df_bt['score_pos'] (rojo) y df_bt['score_neg'] (verde) por período
#   - Sombreado entre las dos líneas (la brecha = discriminación)
# Panel 3: Valor neto realizado por trimestre + acumulado
#   - Barras por trimestre (con el color del año)
#   - Línea punteada de valor neto acumulado

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Backtesting del Modelo SeguraMax en Producción — Trimestral 2022–2024',
             fontweight='bold')

periodos = df_bt['periodo'].tolist()
idx      = range(len(periodos))
colores_anio = {'2022':'#0D7490','2023':'#D97706','2024':'#DC2626'}

# Panel 1
ax = axes[0]
# --- TU CÓDIGO AQUÍ ---

# Panel 2
ax = axes[1]
# --- TU CÓDIGO AQUÍ ---

# Panel 3
ax = axes[2]
# --- TU CÓDIGO AQUÍ ---

plt.tight_layout()
plt.savefig('ej1_backtesting.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔧 6.3 — Diagnóstico del backtesting

Responde las siguientes preguntas con base en los resultados obtenidos.

In [ ]:
# 🔧 COMPLETA: diagnóstico automático del backtesting
# Calcula e imprime:
# 1. ¿Se degradó el AUC? Compara AUC promedio de 2022 vs 2024
# 2. ¿Disminuyó la separación de scores? Compara (score_pos - score_neg) en 2022 vs 2024
# 3. ¿El valor neto fue siempre positivo? ¿Cuántos trimestres tuvieron pérdida?
# 4. ¿El Brier Score empeoró? (indica peor calibración)

print("=" * 60)
print("  DIAGNÓSTICO DEL BACKTESTING — Modelo SeguraMax")
print("=" * 60)

# 🔧 CALCULA los 4 diagnósticos
# --- TU CÓDIGO AQUÍ ---

# 1. Degradación del AUC
auc_2022 = ...
auc_2024 = ...
print(f"\n1. DEGRADACIÓN DEL AUC:")
print(f"   AUC promedio 2022: {auc_2022:.4f}")
print(f"   AUC promedio 2024: {auc_2024:.4f}")
# Agrega tu interpretación

# 2. Separación de scores
sep_2022 = ...  # (score_pos - score_neg).mean() para 2022
sep_2024 = ...
print(f"\n2. SEPARACIÓN DE SCORES (discriminación):")
print(f"   Separación 2022: {sep_2022:.4f}")
print(f"   Separación 2024: {sep_2024:.4f}")

# 3. Valor neto
vnet_total = df_bt['vnet'].sum()
trim_neg   = (df_bt['vnet'] < 0).sum()
print(f"\n3. VALOR NETO REALIZADO:")
print(f"   Total acumulado: ${vnet_total:,.0f}")
print(f"   Trimestres con pérdida: {trim_neg}/{len(df_bt)}")

# 4. Calibración
brier_2022 = ...
brier_2024 = ...
print(f"\n4. CALIBRACIÓN (Brier Score — menor = mejor):")
print(f"   Brier 2022: {brier_2022:.4f}")
print(f"   Brier 2024: {brier_2024:.4f}")

print("\n¿Qué recomiendas al director de Retención?")

---
## Parte 7 — Preguntas de reflexión y decisión final

### 📝 7.1

**a)** El Walk-Forward (Parte 4) y el backtesting (Parte 6) responden preguntas distintas.
¿Cuál es la diferencia entre "el modelo tiene AUC=0.72 en Walk-Forward 2024" y
"el modelo generó un AUC=0.68 en el backtesting del Q3-2024"?

> _Escribe tu respuesta aquí_

---

**b)** El backtesting muestra que en ciertos trimestres el modelo tuvo Recall bajo
(no detectó muchos de los clientes que sí se fueron). ¿Qué le dirías al director
de Retención sobre los clientes que se fueron sin recibir llamada?
¿El modelo tiene la culpa o el umbral τ?

> _Escribe tu respuesta aquí_

---

**c)** El AUC del backtesting cae de 2022 a 2024 pero el valor neto acumulado
sigue siendo positivo. ¿Eso significa que el modelo no necesita re-entrenarse?
¿Qué criterio usarías para decidir cuándo re-entrenar?

> _Escribe tu respuesta aquí_

---

**d) BONUS — Ventana óptima de re-entrenamiento:**
Para predecir 2024, compara tres estrategias de train:
- Largo: 2020–2023 (todos los datos históricos)
- Reciente: 2022–2023 (solo los 2 últimos años)
- Último año: solo 2023

¿Cuál da el AUC más alto? ¿Cuál da el mayor valor neto? ¿Coinciden?

```python
# 🔧 BONUS
mask_test_24 = df['anio'] == 2024
X_te24 = df.loc[mask_test_24, FEATURES].values
y_te24  = df.loc[mask_test_24, 'no_renueva'].values

for nombre, anios_tr in [
    ("Largo (2020-2023)",   [2020,2021,2022,2023]),
    ("Reciente (2022-2023)",[2022,2023]),
    ("Último año (2023)",   [2023]),
]:
    mask_tr = df['anio'].isin(anios_tr)
    # --- TU CÓDIGO AQUÍ ---
    # Entrena, predice, calcula AUC y valor neto
    pass
```